# Corporate Action Event Examples

NSE corporate actions come in several flavours — splits, bonuses, dividends, mergers, rights issues.
Each type requires different handling in a price series or backtesting system.

This notebook shows:
- The full corporate action taxonomy used by TickerTruth
- How to read split and bonus ratio strings
- How `confidence_flag` tells you how much to trust each event
- How a single security (RELIANCE) accumulates multiple actions over time

**Data source:** `sample_data/corporate_actions.parquet`

In [ ]:
import pandas as pd
from pathlib import Path

_cwd = Path.cwd()
SAMPLE_DATA = (
    _cwd / "sample_data"
    if (_cwd / "sample_data").exists()
    else _cwd.parent / "sample_data"
)

ca = pd.read_parquet(SAMPLE_DATA / "corporate_actions.parquet")
ca["ex_date"] = pd.to_datetime(ca["ex_date"])

print(f"Total events: {len(ca)}")
print(f"Columns: {ca.columns.tolist()}")

## Action type distribution

In [ ]:
dist = (
    ca.groupby("action_type")
    .agg(
        events=("symbol", "count"),
        symbols=("symbol", "nunique"),
    )
    .sort_values("events", ascending=False)
)
print(dist.to_string())

## Reading ratio strings

`value_ratio` stores the event ratio as a string (`1:5`, `1:2`) for splits and bonuses,
or as a numeric string (`21.0`) for cash dividends.

**Split 1:5** — one existing share becomes five shares; price falls to ~1/5 of pre-split level.  
**Bonus 1:2** — two bonus shares issued per one held; price falls to ~1/3 of pre-bonus level.

The `adjustment_factor` column gives the pre-computed multiplier to apply to historical prices.

In [ ]:
# Show one example of each type
for action_type in ["SPLIT", "BONUS", "DIVIDEND"]:
    row = ca[ca["action_type"] == action_type].iloc[0]
    print(f"--- {action_type} ---")
    print(f"  Symbol:            {row['symbol']}")
    print(f"  Ex-date:           {row['ex_date'].date()}")
    print(f"  Ratio/Value:       {row['value_ratio']}")
    print(f"  Adjustment factor: {row['adjustment_factor']}")
    if action_type == "DIVIDEND":
        print(f"  Dividend amount:   INR {row['dividend_amount']}")
        print(f"  Frequency:         {row['frequency']}")
    print()

## Confidence flags

Each event carries a `confidence_flag` — a data-quality signal:

| Flag | Meaning |
|---|---|
| `HIGH` | Official NSE source, all fields present and consistent |
| `MEDIUM` | Value present but a secondary field (frequency, record date) is ambiguous |
| `LOW` | Ratio estimated or source is a scrape with partial data |
| `UNRESOLVED` | Conflicting data from two or more sources; do not use in production |

In [ ]:
flag_dist = ca.groupby(["confidence_flag", "action_type"]).size().unstack(fill_value=0)
print(flag_dist.to_string())
print()
high_pct = (ca["confidence_flag"] == "HIGH").mean() * 100
print(f"{high_pct:.0f}% of events are HIGH confidence in this sample.")

## RELIANCE — full corporate action history

RELIANCE has had multiple splits and bonuses over the period covered by this sample.
Each event compounds with previous ones — you must apply them in chronological order.

In [ ]:
rel = ca[ca["symbol"] == "RELIANCE"].sort_values("ex_date").copy()
rel["ex_date"] = rel["ex_date"].dt.date
print(
    rel[
        [
            "ex_date",
            "action_type",
            "value_ratio",
            "adjustment_factor",
            "confidence_flag",
        ]
    ].to_string(index=False)
)

## Cumulative effect of chained events

Each event's adjustment factor must be *multiplied* together to get the total
restated-price multiplier from the beginning of the series to today.

In [ ]:
# Only price-affecting events (not dividends)
price_events = rel[rel["action_type"].isin(["SPLIT", "BONUS"])].copy()
price_events["cumulative_factor"] = price_events["adjustment_factor"].cumprod().round(6)
print(
    price_events[
        [
            "ex_date",
            "action_type",
            "value_ratio",
            "adjustment_factor",
            "cumulative_factor",
        ]
    ].to_string(index=False)
)
print()
total = price_events["adjustment_factor"].prod()
print(f"Total adjustment from {price_events['ex_date'].min()} to present: {total:.4f}")
print(
    f"A price of INR 1,000 pre-{price_events['ex_date'].min()} equals INR {1000 * total:.2f} on a fully-adjusted basis."
)

---
## Key takeaway

- Always filter to `confidence_flag IN ('HIGH', 'MEDIUM')` for production use.
- Apply adjustment factors in **chronological order** — the cumulative product is not commutative once you mix splits and bonuses.
- Dividends (`adjustment_factor = 1.0`) do not affect price series continuity; they are recorded for completeness and for dividend-adjusted return calculations.